# Notebook 4 · So what? and what else did we learn? (Questions 4 and 5)

**Question 4:** What should leadership take away, and which two or three SBLCs should the client watch most closely?
**Question 5:** What other interesting insights came out of the data?

**Approach**
1. Build **one scorecard** per SBLC: scale, growth, credit quality, credit trend.
2. Label each lender with simple, stated rules.
3. Choose the **watchlist by explicit rules** (not by gut), then test how sensitive it is to the thresholds.
4. Collect the facts behind the key insights in one place.
5. Write the takeaways for leadership.

**Inputs:** the output files of Notebooks 2 and 3, plus the clean loan file.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

OUT = Path("../../assignment1/outputs")
growth = pd.read_csv(OUT / "sblc_growth_windows.csv", index_col=0)          # Notebook 2
profile = pd.read_csv(OUT / "sblc_profile.csv", index_col=0)                # Notebook 2
credit = pd.read_csv(OUT / "sblc_credit_scorecard.csv", index_col=0)        # Notebook 3
half = pd.read_csv(OUT / "sblc_credit_by_half_decade.csv", index_col=0)     # Notebook 3
drivers = pd.read_csv(OUT / "credit_drivers_sblc_vs_others.csv", index_col=0)
market = pd.read_csv(OUT / "sblc_market_by_year.csv", index_col=0)

df = pd.read_parquet("../../data/processed/loans_clean.parquet")
print("Lenders in scorecard:", len(credit), "| growth rows:", len(growth))

Lenders in scorecard: 12 | growth rows: 12


## Step 1 · One scorecard
We combine growth (Notebook 2) and credit (Notebook 3):
- **pool_share_late_%:** the lender's share of all SBLC dollars in FY2017-19, which measures scale.
- **share_change_bps:** change in its share of all 7(a) dollars (basis points).
- **obs_exp:** observed / expected charge-offs (1.0 = same as other lenders for comparable loans), with its 95% interval.
- **credit_trend:** change in obs_exp from FY2010-14 loans to FY2015-19 loans (positive = getting worse).

In [2]:
sc = pd.DataFrame({
    "pool_share_late_%": growth["pool_share_late_%"],
    "share_change_bps": growth["share_change_bps"],
    "growth_multiple": growth["gross_late_m"] / growth["gross_early_m"].replace(0, np.nan),
    "loans": credit["loans"],
    "obs_exp": credit["obs_exp"],
    "obs_exp_lo": credit["obs_exp_lo"],
    "obs_exp_hi": credit["obs_exp_hi"],
    "n_charge_offs": credit["n_bad"],
    "loss_rate_$": credit["loss_rate_$"],
    "obs_exp_early": half["FY2010-14"],
    "obs_exp_late": half["FY2015-19"],
})
sc["credit_trend"] = sc["obs_exp_late"] - sc["obs_exp_early"]
sc = sc.sort_values("pool_share_late_%", ascending=False)
sc.round(2)

,pool_share_late_%,share_change_bps,growth_multiple,loans,obs_exp,obs_exp_lo,obs_exp_hi,n_charge_offs,loss_rate_$,obs_exp_early,obs_exp_late,credit_trend
lender,,,,,,,,,,,,
Newtek,41.22,124.73,4.86,2874,1.49,1.32,1.67,247,0.04,2.11,1.30,-0.81
Harvest,18.06,72.49,12.80,478,0.23,0.09,0.59,4,0.01,0.00,0.27,0.27
Readycap,17.61,7.90,1.86,975,1.26,0.99,1.59,62,0.03,0.82,1.64,0.82
Centerstone,7.66,35.37,NaN,205,0.00,0.00,0.48,0,0.00,NaN,0.00,NaN
First Western,6.18,1.50,1.77,295,0.46,0.21,0.98,6,0.01,0.47,0.43,-0.03
CRF,2.76,9.19,5.99,494,1.41,1.06,1.87,43,0.05,1.44,1.40,-0.04
Velocity,2.47,-0.91,1.55,5008,2.75,2.63,2.88,1330,0.10,2.76,1.83,-0.93
Hana,1.51,-55.54,0.19,671,0.97,0.71,1.32,38,0.02,1.07,0.78,-0.29
Grow America,1.45,-2.84,1.18,268,1.00,0.63,1.57,17,0.05,0.97,1.03,0.06


## Step 2 · Label each lender with simple rules
**Growth label:** Gaining (share up 25+ bps), Losing (down 25+ bps), otherwise Holding.

**Credit label:**
- *Strong (thin record):* fewer than 10 charge-offs and below-average rate. Good, but too little evidence to be sure.
- *Better than market:* the whole 95% interval is below 1.0.
- *Worse than market:* the whole interval is above 1.3.
- *In line:* ratio 1.1 or below. *Slightly worse:* anything else.

In [3]:
def growth_label(bps):
    if bps >= 25:  return "Gaining"
    if bps <= -25: return "Losing"
    return "Holding"

def credit_label(r):
    if r.n_charge_offs < 10 and r.obs_exp < 1: return "Strong (thin record)"
    if r.obs_exp_hi < 1:   return "Better than market"
    if r.obs_exp_lo > 1.3: return "Worse than market"
    if r.obs_exp <= 1.1:   return "In line"
    return "Slightly worse"

sc["growth"] = sc["share_change_bps"].map(growth_label)
sc["credit"] = sc.apply(credit_label, axis=1)
display(sc[["pool_share_late_%", "share_change_bps", "growth", "obs_exp", "n_charge_offs", "credit"]].round(2))

arch = sc.reset_index().rename(columns={"index": "lender"})
fig = px.scatter(arch, x="share_change_bps", y="obs_exp", size="pool_share_late_%", color="credit", text="lender",
                 size_max=55, labels={"share_change_bps": "Change in market share (bps), FY2010-12 to FY2017-19",
                                      "obs_exp": "Observed / expected charge-offs (1 = average)"},
                 title="Growth vs credit quality: where each SBLC sits (bubble = share of SBLC dollars)")
fig.add_hline(y=1, line_dash="dash"); fig.add_vline(x=0, line_dash="dash")
fig.update_traces(textposition="top center"); fig.update_layout(template="plotly_white", height=560)
fig.show()

,pool_share_late_%,share_change_bps,growth,obs_exp,n_charge_offs,credit
lender,,,,,,
Newtek,41.22,124.73,Gaining,1.49,247,Worse than market
Harvest,18.06,72.49,Gaining,0.23,4,Strong (thin record)
Readycap,17.61,7.90,Holding,1.26,62,Slightly worse
Centerstone,7.66,35.37,Gaining,0.00,0,Strong (thin record)
First Western,6.18,1.50,Holding,0.46,6,Strong (thin record)
CRF,2.76,9.19,Holding,1.41,43,Slightly worse
Velocity,2.47,-0.91,Holding,2.75,1330,Worse than market
Hana,1.51,-55.54,Losing,0.97,38,In line
Grow America,1.45,-2.84,Holding,1.00,17,In line


## Step 3 · Which SBLCs should the client watch?
The client should watch lenders that **matter** to its plan. We use three explicit flags:
1. **Big:** at least 15% of SBLC dollars (they set the competitive pace).
2. **Fast:** gained at least 25 bps of 7(a) share (they are taking business).
3. **Credit flag:** observed / expected of 1.25 or more (their growth may carry risk).

A lender with **two or more flags** goes on the watchlist. The thresholds are judgment, so the next step tests how much they matter.

In [4]:
BIG, FAST, CREDIT_HI = 15, 25, 1.25     # thresholds (judgment; tested for sensitivity below)

flags = pd.DataFrame({
    "Big (>=15% of SBLC $)": sc["pool_share_late_%"] >= BIG,
    "Fast (>=+25 bps share)": sc["share_change_bps"] >= FAST,
    "Credit flag (obs/exp >=1.25)": sc["obs_exp"] >= CREDIT_HI,
})

flags["n_flags"] = flags.sum(axis=1)
watch = flags.sort_values("n_flags", ascending=False)
display(watch)

,Big (>=15% of SBLC $),Fast (>=+25 bps share),Credit flag (obs/exp >=1.25),n_flags
lender,,,,
Newtek,True,True,True,3
Harvest,True,True,False,2
Readycap,True,False,True,2
Centerstone,False,True,False,1
CRF,False,False,True,1
Velocity,False,False,True,1
Lendistry,False,False,True,1
CapitalSpring,False,False,True,1
First Western,False,False,False,0


### How stable is the watchlist?
We rerun the rule for 27 combinations of thresholds (Big at 10/15/20%, Fast at 20/25/35 bps, credit flag at 1.2/1.25/1.4) and count how often each lender lands on the watchlist.

In [6]:
rows = []
for big in (10, 15, 20):
    for fast in (20, 25, 35):
        for ch in (1.2, 1.25, 1.4):
            f = ((sc["pool_share_late_%"] >= big).astype(int) + (sc["share_change_bps"] >= fast).astype(int)
                 + (sc["obs_exp"] >= ch).astype(int))
            rows.append(f[f >= 2].index.tolist())
freq = pd.Series([n for r in rows for n in r]).value_counts().rename("times on watchlist (of 27 settings)")
display(freq.to_frame())

,times on watchlist (of 27 settings)
Newtek,27
Harvest,18
Readycap,12


## Step 4 · The facts behind the insights (Question 5)
One table of the numbers quoted in the takeaways, so every claim can be traced to code.

In [7]:
SHORT = {
    "Newtek Small Business Finance, Inc.": "Newtek", "VelocitySBA, LLC": "Velocity",
    "Readycap Lending, LLC": "Readycap", "Hana Small Business Lending, Inc.": "Hana",
    "Harvest Small Business Finance, LLC": "Harvest", "First Western SBLC, Inc": "First Western",
    "CRF Small Business Loan Company, LLC": "CRF", "Centerstone SBA Lending, Inc.": "Centerstone",
    "Grow America Fund, Incorporated": "Grow America", "Lendistry SBLC, LLC": "Lendistry",
    "Loan Source Incorporated": "Loan Source", "CapitalSpring SBLC, LLC": "CapitalSpring",
}
df["is_sblc"] = df["bankname"].isin(SHORT)
df["lender"] = df["bankname"].map(SHORT)
df["bad"] = df["charged_off_ever"]
s, others = df[df["is_sblc"]], df[~df["is_sblc"]]
ex_v = s[s["lender"] != "Velocity"]
months = (df["chargeoffdate"] - df["approvaldate"]).dt.days / 30.4

facts = {}
facts["SBLC ever-charged-off %"] = s["bad"].mean() * 100
facts["Other lenders ever-charged-off %"] = others["bad"].mean() * 100
facts["SBLC ex-Velocity %"] = ex_v["bad"].mean() * 100
facts["Velocity share of SBLC charge-offs %"] = s.loc[s["lender"] == "Velocity", "bad"].sum() / s["bad"].sum() * 100
facts["SBLC $ loss rate %"] = s["grosschargeoffamount"].sum() / s["grossapproval"].sum() * 100
facts["Other $ loss rate %"] = others["grosschargeoffamount"].sum() / others["grossapproval"].sum() * 100

n_status = (df["loanstatus"] == "CHGOFF").sum(); n_ever = df["bad"].sum()
facts["Charge-offs missed by status-only definition %"] = (n_ever / n_status - 1) * 100

old = df[(df["bad"] == 1) & df["approvalfy"].between(2010, 2012)]
facts["Share of FY2010-12 charge-offs that appeared after month 78 %"] = (months[old.index] > 78).mean() * 100

facts["SBLC loans sold on secondary market %"] = (s["soldsecmrktind"] == "Y").mean() * 100
facts["Other lenders' loans sold %"] = (others["soldsecmrktind"] == "Y").mean() * 100

plp = df[df["processingmethod"].isin(["Preferred Lenders Program", "Preferred Lenders with EWCP"])]
facts["PLP charge-off % SBLC"] = plp.loc[plp["is_sblc"], "bad"].mean() * 100
facts["PLP charge-off % others"] = plp.loc[~plp["is_sblc"], "bad"].mean() * 100

v = s[s["lender"] == "Velocity"]
v_early, v_late = v[v["approvalfy"] <= 2012], v[v["approvalfy"] >= 2017]
facts["Velocity loans FY10-12"] = len(v_early); facts["Velocity median loan FY10-12 $"] = v_early["grossapproval"].median()
facts["Velocity loans FY17-19"] = len(v_late);  facts["Velocity median loan FY17-19 $"] = v_late["grossapproval"].median()

hhi = lambda x: ((x / x.sum() * 100) ** 2).sum()
facts["HHI early"] = hhi(growth["gross_early_m"]); facts["HHI late"] = hhi(growth["gross_late_m"])
facts["SBLC share of 7(a) $ FY2010-12 %"] = growth["gross_early_m"].sum() / (market.loc[2010:2012, "mkt_gross"].sum() / 1e6) * 100
facts["SBLC share of 7(a) $ FY2017-19 %"] = growth["gross_late_m"].sum() / (market.loc[2017:2019, "mkt_gross"].sum() / 1e6) * 100
display(pd.Series(facts).round(1).to_frame("value"))

,value
SBLC ever-charged-off %,15.5
Other lenders ever-charged-off %,7.6
SBLC ex-Velocity %,6.9
Velocity share of SBLC charge-offs %,74.8
SBLC $ loss rate %,3.3
Other $ loss rate %,2.9
Charge-offs missed by status-only definition %,20.8
Share of FY2010-12 charge-offs that appeared after month 78 %,26.8
SBLC loans sold on secondary market %,98.2
Other lenders' loans sold %,24.9


## Save the final scorecard

In [8]:
final = sc[["pool_share_late_%", "share_change_bps", "growth", "obs_exp", "obs_exp_lo", "obs_exp_hi",
            "credit_trend", "n_charge_offs", "credit", "loans"]].join(watch[["n_flags"]])
final.to_csv(OUT / "sblc_final_scorecard.csv")
pd.Series(facts).to_csv(OUT / "key_facts.csv", header=["value"])
print("saved")

saved


## Question 4 · So what? Takeaways for leadership

*Assumption: the client is a commercial SBLC making mid-to-large PLP-type loans. If it is a small-loan lender, the Velocity and Lendistry findings matter more.*

**1. The non-bank segment is winning share and consolidating.** SBLCs went from about 2.7% to about 4.6% of 7(a) dollars (FY2010-12 to FY2017-19). Concentration rose (HHI about 2,000 to about 2,450), and Newtek alone holds about 41% of SBLC dollars. The client is competing against a segment that is growing faster than the market, but dominated by a few players.

**2. Growth does not have to cost credit quality.** Newtek's credit ratio improved from about 2.1 to about 1.3 while its loan count grew about 3.4x. Readycap shows the opposite: its ratio worsened from about 0.8 to about 1.6 as volume nearly doubled.

**3. The credit gap is about product, not lender type.** On PLP loans, SBLCs charge off 6.3% of loans, exactly like other lenders (6.3%). The gap sits in small, unsecured Express loans.

**4. The three SBLCs to watch**
- **Newtek**: the scale leader (about 41% of SBLC dollars, +125 bps of 7(a) share). Credit is above average (about 1.5x) but improving. It sets the competitive benchmark. *Watchlist in 27 of 27 tests.*
- **Harvest**: the fastest relative gainer (about 11 to about 83 bps) with very clean credit (4 charge-offs in 478 loans). But its record is young, so the question is whether it holds as loans age. *Watchlist in 18 of 27 tests.*
- **Readycap**: the other large lender (about 18% of SBLC dollars) holding share, but with credit that deteriorated as it grew. It is the early-warning case for growth bought with risk. *Watchlist in 12 of 27 tests, the least certain pick.*

**5. Implications for the client's plan**
- Use **6.3% ever-charged-off on PLP loans** as the credit benchmark to defend, not the headline SBLC average.
- Large-ticket, collateralised lending (Harvest, First Western, Centerstone) is where the cleanest records sit, but expect charge-offs to rise as those loans age.
- Avoid tiny unsecured Express volume unless pricing covers losses near 27% of loans.
- Track Newtek's share and Readycap's credit trend as early signals.
- Run the same observed/expected analysis on the client's own book. We cannot do this without its data.

## Question 5 · Other interesting insights
1. **The headline "SBLCs are twice as risky" is mostly one lender.** Velocity accounts for about 75% of all SBLC charge-offs. Without it, SBLCs charge off 6.9% vs 7.6% for other lenders.
2. **Loan count and dollars tell different stories.** By count, SBLCs charge off 15.5% vs 7.6%. By dollars, 3.3% vs 2.9%, because Velocity's losses are on tiny loans.
3. **Status-based default counts miss about 21% of charge-offs.** Loans charged off and later reclassified (paid in full, closed) still carry a charge-off amount.
4. **Losses arrive late.** The median charge-off takes about 55 months. About 27% of charge-offs on FY2010-12 loans appeared after month 78, so the newest vintages (and young lenders such as Harvest and Centerstone) will look worse over time.
5. **SBLCs sell nearly everything they originate.** About 98% of SBLC loans were sold on the secondary market, against about 25% for other lenders. This is consistent with an originate-and-sell model.
6. **Winning can mean changing your business.** Velocity went from 4,643 loans at a $10k median (FY2010-12) to 67 loans at a $788k median (FY2017-19).
7. **Data traps:** `bankname` is the lender the loan is *currently assigned to*, the FDIC appears as a lender (it took over loans from failed banks), and one SBLC license can sit behind several brand names.

## Limits of this analysis
- Several lenders have too few loans or charge-offs for firm conclusions (Centerstone, Harvest, First Western, Loan Source, CapitalSpring).
- The benchmark is mostly banks, whose loan mix differs from SBLCs.
- The watchlist thresholds are judgment. The sensitivity test shows Newtek is robust and Readycap is not.
- Approvals show who is lending, not who is lending profitably: we have no pricing, cost or profit data.